In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
%matplotlib inline

In [ ]:
# The growth-curve fitting is done by tecantaloupe, which is not on PyPI and is not vendored
# in this repository. Clone it and point TECANTALOUPE_DIR at the checkout:
#
#   git clone https://github.com/flamholz/tecantaloupe
#   git -C tecantaloupe checkout 8199fbb48e4764f96c172de7009dc70c229ae4e6
#
# It needs pandas < 2 (it calls DataFrame.iteritems, removed in pandas 2.0); installation.md
# gives the pinned environment this notebook is run in.
import os
import sys

_tecantaloupe = os.environ.get("TECANTALOUPE_DIR")
if not _tecantaloupe or not os.path.isdir(_tecantaloupe):
    raise RuntimeError(
        "Set TECANTALOUPE_DIR to a clone of https://github.com/flamholz/tecantaloupe "
        "(tested at 8199fbb48e4764f96c172de7009dc70c229ae4e6). See installation.md."
    )
sys.path.append(_tecantaloupe)
from growth.plate_spec import PlateSpec
from growth.plate_time_course_parser import SavageLabM1000Excel


## Replicate 20250220

I replicated the experiment

In [ ]:
plate_spec_file = '20250220_plate_spec.csv'
ps = PlateSpec.FromFile(plate_spec_file)
name_mapping = ps.well_to_name_mapping()

parser = SavageLabM1000Excel()
timecourse = parser.ParseFromFilename('20250220_Results_Autosave.xlsx')

blanked = timecourse.blank()
smoothed = blanked.smooth()

OD_data = smoothed.data_for_label('A600')
means = smoothed.mean_by_name(ps)
sems = smoothed.sem_by_name(ps)

mean_OD = means.data_for_label('A600')
sems_OD = sems.data_for_label('A600')

In [ ]:
lag_times = means.LagTime(density_label='A600')
growth_rates = means.MaxGrowthRates(density_label='A600')
yields = means.GrowthYield(density_label='A600')

In [ ]:
pos_set = [c for c in mean_OD.columns if c.startswith('WT') or c.startswith('HCA')]
neg_set = [c for c in mean_OD.columns if c.startswith('D152N')]
exp_set = [c for c in mean_OD.columns if c.startswith('seq')]

pos = [p for p in pos_set if '200' in p]
neg = [p for p in neg_set if '200' in p]
exp = [p for p in exp_set if '200' in p]

In [ ]:
labels = sorted(pos) + sorted(exp) + sorted(neg)
colors = ['green'] * len(pos) + ['blue'] * len(exp) + ['red'] * len(neg)
axis_labels = [l[:l.find('+')].replace('_unsorted','').replace('_sorted','') for l in labels]

ylds = [yields[l] for l in labels]

figure = plt.figure(figsize=(8,4))
xs = np.arange(len(ylds))
plt.bar(xs, ylds, color=colors)
plt.ylabel('Growth Yield (OD600 Units)')
plt.xticks(xs, axis_labels, rotation=90)
plt.show()

In [ ]:
labels = sorted(pos) + sorted(exp) + sorted(neg)
grs = [growth_rates[l] for l in labels]

figure = plt.figure(figsize=(8,4))
xs = np.arange(len(grs))
plt.bar(xs, grs, color=colors)
plt.ylabel('Max Growth Rate (/hr)')
plt.xticks(xs, axis_labels, rotation=90)
plt.show()

In [ ]:
exp_to_plot = list(filter(lambda x: any([y in x for y in ['1789', '1802', '1363', '1251', '1360', '1944', '1729', '1670']]), exp))

In [ ]:
colors = sns.color_palette('Set2')
time_h = mean_OD.time_s / (60*60)
fig, axs = plt.subplots(figsize=(10,6))

sns.set_theme(style='white', font_scale=1.7)

for i, c in enumerate(sorted(pos)):
    label = c.replace('_unsorted', '').replace('_sorted', '').replace('+ 200 ATC', '')
    color = 'green'
    plt.fill_between(time_h, mean_OD[c] - sems_OD[c], mean_OD[c] + sems_OD[c],
                     color=color, alpha=0.5)
    plt.plot(time_h, mean_OD[c], label=label, color=color, figure=fig)


for i, c in enumerate(exp_to_plot):
    label = c.replace('_unsorted', '').replace('_sorted', '').replace('+ 200 ATC', '')
    color = colors[i % len(colors)]
    print(color)
    plt.fill_between(time_h, mean_OD[c] - sems_OD[c], mean_OD[c] + sems_OD[c],
                     color=color, alpha=0.5)
    plt.plot(time_h, mean_OD[c], label=label, color=color, figure=fig)

for i, c in enumerate(sorted(neg)):
    label = c.replace('_unsorted', '').replace('_sorted', '').replace('+ 200 ATC', '')
    color = 'red'
    plt.fill_between(time_h, mean_OD[c] - sems_OD[c], mean_OD[c] + sems_OD[c],
                     color=color, alpha=0.5)
    plt.plot(time_h, mean_OD[c], label=label, color=color, ls='--', figure=fig)

plt.xlabel('Time (hours)')
plt.ylabel('OD600')
plt.title(r'$\beta$-Carbonic Anhydrase Growth Curves')
plt.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.tight_layout()
plt.savefig('Growth_Curves.pdf')
plt.savefig('Growth_Curves.png')